<a href="https://colab.research.google.com/github/Nitsh-kumar/LangChain_learning/blob/main/Langchain_learning_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prompt Template and Structured Output


Here, we will learn about the different prompt templates and output parsers.

Resources :https://www.youtube.com/watch?v=OCpTAVGO3IQ

In [1]:
# Basic installation of the packages
%pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.7 MB/s eta 0:00:00


In [42]:
#importing the necessary packages

import os
from google.colab import userdata
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import JsonOutputParser
from langchain_groq import ChatGroq
from pprint import pprint
from langsmith import traceable

In [54]:
# Setting up the environment variables and API's


#LLM
os.environ['GROQ_API_KEY']  = userdata.get('GROQ_API_KEY')

#Langsmith for learning and tracking
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('langsmith_api')
# os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = 'Learning_langsmith'

In [45]:
# setting up the llm model

model = ChatGroq(
    model = 'llama-3.1-8b-instant',
    temperature= 0.3,
    streaming= True,
)

Note: If you do not know about how to check for any modules or classes in langchain, there is a common way.


import package_name as p


then dir(p)


and if you want a specific class then





from langchain_core.prompts import PromptTemplate






help(PromptTemplate)



In [23]:
# First Technique or template is PromptTemplate
query_prompt = PromptTemplate(
    template="""
                You are a query classifier.
                Classify the user query into one of:
                - billing
                - technical_support
                - general

                User query:
                {query}
                Return ONLY the category name.
                """,
    input_variables=["query"],
)
prompt = query_prompt.format(query = 'My invoice is not having a correct address')
pprint(prompt)

('\n'
 '                You are a query classifier.\n'
 '                Classify the user query into one of:\n'
 '                - billing\n'
 '                - technical_support\n'
 '                - general\n'
 '\n'
 '                User query:\n'
 '                My invoice is not having a correct address\n'
 '                Return ONLY the category name.\n'
 '                ')


In [24]:
model_response  = model.invoke(prompt)

In [25]:
pprint(model_response.content)

'billing'


In [27]:
# Instead of directly invoking the model we can build a small pipeline by using pipe operation
pipeline  = query_prompt | model

In [28]:
response  = pipeline.invoke({"query": "laptop is not good"})
pprint(response.content)

'technical_support'


# 🧠 LangChain PromptTemplate — Notes

---

## 1️⃣ What PromptTemplate Is

**PromptTemplate** is a **deterministic prompt constructor**.

- Converts structured inputs into a prompt in a predictable way  
- Removes manual string concatenation  
- Enforces explicit input variables  

👉 Think of it as a **prompt contract**, **not** a model caller.

---

## 2️⃣ Why PromptTemplate Still Exists in Modern LangChain

PromptTemplate is still relevant because **production systems require**:

- **Repeatability**  
  Same input → same prompt  

- **Safety**  
  No hidden variables or accidental prompt injection  

- **Traceability**  
  Prompts can be logged, replayed, and audited  

- **Versioning**  
  Prompts can be diffed, rolled back, and A/B tested  

- **Reusability**  
  Same prompt works across APIs, batch jobs, and evaluations  

These properties are **critical in enterprise and regulated environments**.

---

## 3️⃣ What PromptTemplate Does *NOT* Do

PromptTemplate:

- ❌ Does **not** call the LLM  
- ❌ Does **not** manage execution  
- ❌ Does **not** handle model selection  

✅ Its **only responsibility** is **prompt construction**.

---

## 4️⃣ Where PromptTemplate Fits in the Modern LangChain Stack

Modern LangChain separates responsibilities clearly:

- **PromptTemplate** → builds deterministic prompts  
- **ChatPromptTemplate** → structures multi-message chat prompts  
- **LLM / Chat Model** → executes prompts  
- **LCEL (pipe operator)** → connects components cleanly  

👉 PromptTemplate is a **low-level building block**, not the orchestrator.

---

## 5️⃣ Why Industry Prefers PromptTemplate Over Raw Strings

### Raw strings:
- Hard to validate  
- Break silently when variables change  
- Difficult to audit and test  

### PromptTemplate:
- Enforces explicit inputs  
- Prevents missing or extra variables  
- Produces consistent prompts across environments  

👉 Industry prefers **boring and predictable** over clever.

---

## 6️⃣ Why PromptTemplate Lives in `langchain_core`

Prompts are inherently:

- **Model-agnostic**  
- **Stateless**  
- **Deterministic**  

These properties make them **core primitives**, which is why they live in  
`langchain_core` instead of model-specific packages.

---

## 7️⃣ PromptTemplate vs ChatPromptTemplate

### PromptTemplate
- Single text prompt  
- Best for:
  - Classification  
  - Summarization  
  - Extraction  
  - RAG formatting  

### ChatPromptTemplate
- Multi-message structure (system, human, assistant)  
- Best for:
  - Conversational systems  
  - Chat-based UX  

👉 Internally, **ChatPromptTemplate still relies on PromptTemplate-like behavior**.

---

## 8️⃣ How Invocation Works (Conceptually)

Execution flow:

1. Inputs are passed once  
2. PromptTemplate formats the prompt  
3. The LLM consumes the formatted prompt  
4. A response is returned  

Using **LCEL** allows this to happen in **one logical step** while keeping responsibilities separated.

---

## 9️⃣ Why LCEL (“Pipe” Pattern) Is Important

LCEL:

- Removes glue code  
- Makes pipelines readable  
- Allows easy extension (parsers, tools, routers)  
- Replaces deprecated abstractions like `LLMChain`  

Industry prefers LCEL because it is:

- **Composable**  
- **Testable**  
- **Future-proof**  

---

## 🔟 When PromptTemplate Is Usually *NOT* Used

PromptTemplate is often skipped during:

- Rapid prototyping  
- Exploratory notebooks  
- Prompt experimentation  

👉 Once a prompt stabilizes and moves toward production,  
**PromptTemplate is usually introduced**.

---

## 1️⃣1️⃣ Common Mistakes to Avoid

- Assuming PromptTemplate calls the model  
- Using outdated imports or tutorials  
- Mixing global Python installs with venv kernels  
- Treating notebooks and terminals as the same runtime  
- Ignoring prompt versioning in production systems  

---

## 🧠 Final One-Line Summary

> **PromptTemplate is a deterministic, versionable prompt contract used in production to separate prompt construction from model execution.**

---


In [33]:
#Now let;s Cover Chat Template
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Explain feature engineering in simple terms.")
])

In [38]:
messages = chat_prompt.format_messages()
pprint(messages)

[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Explain feature engineering in simple terms.', additional_kwargs={}, response_metadata={})]


In [39]:
# if you want to add any variable in the prompt you can do that as well
chat_prompt_2 = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Can you Explain this Topic:{topic} in Simple words in 50-60 words.")
])
messages_2 = chat_prompt_2.format_messages(topic = 'Agents')
pprint(messages_2)


[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Can you Explain this Topic:Agents in Simple words in 50-60 words.', additional_kwargs={}, response_metadata={})]


In [41]:
#you can iterate over the template using the for loop
for msg in messages:
  print(msg.type , "->" ,  msg.content)

system -> You are a helpful assistant.
human -> Explain feature engineering in simple terms.


In [43]:
@traceable(run_type="chain", name="Chat Prompt Pipeline")
def chat_pipeline(topic: str):
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant."),
        ("human", "Explain {topic} in 20 words.")
    ])

    pipeline = prompt | model
    return pipeline.invoke({"topic": topic}).content

In [44]:
result  = chat_pipeline('Feature Engineering')
print(result)

**Feature Engineering: A Key Step in Machine Learning**

Feature engineering is the process of selecting, transforming, and extracting relevant information from raw data to create features that can be used to train a machine learning model. The goal of feature engineering is to create a set of features that are informative, relevant, and useful for the machine learning algorithm to learn from.

**Why is Feature Engineering Important?**

Feature engineering is a crucial step in machine learning because it can significantly impact the performance of a model. A well-engineered set of features can:

1. **Improve model accuracy**: By selecting relevant features, you can reduce the noise in the data and improve the model's ability to learn from it.
2. **Reduce overfitting**: By selecting a subset of relevant features, you can reduce the risk of overfitting, which occurs when a model is too complex and fits the training data too closely.
3. **Increase model interpretability**: By selecting fe

In [46]:
result  = chat_pipeline('Stages in machine learning projects')
print(result)

Machine learning projects typically involve several stages, which are crucial for developing and deploying a successful model. Here are the key stages in a machine learning project:

**1. Problem Definition (Business Understanding)**

- Identify the problem or opportunity that the project aims to address.
- Define the goals and objectives of the project.
- Determine the target audience and their needs.
- Gather business requirements and constraints.

**2. Data Collection (Data Understanding)**

- Gather relevant data from various sources (e.g., databases, APIs, web scraping).
- Clean and preprocess the data to ensure it's in a usable format.
- Explore the data to understand its distribution, patterns, and relationships.
- Identify potential biases and missing values in the data.

**3. Data Preprocessing (Data Preparation)**

- Handle missing values and outliers in the data.
- Transform and normalize the data to ensure consistency.
- Split the data into training, validation, and testing

In [56]:
result  = chat_pipeline('Stages in DATA PIPELINES')
print(result)

Data pipelines are a series of processes that extract, transform, and load (ETL) data from various sources, transform it into a desired format, and load it into a target system. Here are the common stages in a data pipeline:

1. **Data Ingestion**: This stage involves collecting data from various sources, such as databases, APIs, files, or social media platforms. The data is typically in a raw format and may require cleaning and processing before it can be used.

2. **Data Cleaning**: In this stage, the raw data is cleaned and processed to remove any errors, inconsistencies, or irrelevant information. This may involve handling missing values, removing duplicates, and standardizing data formats.

3. **Data Transformation**: This stage involves transforming the cleaned data into a format that is suitable for analysis or storage. This may involve aggregating data, performing calculations, or converting data types.

4. **Data Enrichment**: In this stage, additional data is added to the tra

# 🧠 LangChain ChatPromptTemplate — Notes

---

## 1️⃣ What ChatPromptTemplate Is

**ChatPromptTemplate** is a **structured prompt builder for chat-based LLMs**.

- It organizes prompts as **messages**, not plain text  
- It supports roles like **system**, **human**, and **assistant**  
- It produces a message sequence that chat models understand  

👉 Think of it as a **conversation contract**, not just a prompt.

---

## 2️⃣ Why ChatPromptTemplate Exists (and PromptTemplate Isn’t Enough)

Modern LLMs (GPT-4, GPT-4o, Claude, Gemini) are **chat-native**.

They reason differently based on:
- Message order
- Message roles
- System instructions vs user input

ChatPromptTemplate exists to:
- Preserve **role semantics**
- Prevent instruction confusion
- Make chat behavior predictable

---

## 3️⃣ Core Idea Behind ChatPromptTemplate

A chat prompt is **not one string**.

It is a **sequence of messages**, each with:
- A role
- Content
- Optional variables

ChatPromptTemplate manages this structure safely and consistently.

---

## 4️⃣ Message Roles (Industry Meaning)

### 🧩 System
- Sets global behavior and constraints
- Defines tone, policy, persona
- Highest priority instruction

Used for:
- Safety rules
- Role definition
- Guardrails

---

### 👤 Human
- Represents user input
- Usually dynamic
- Changes per request

Used for:
- Questions
- Commands
- Tasks

---

### 🤖 Assistant
- Represents prior model responses
- Used mainly in:
  - Multi-turn conversations
  - Memory-based systems

---

## 5️⃣ Why Industry Uses ChatPromptTemplate

ChatPromptTemplate is preferred because it provides:

- **Instruction hierarchy**
  - System > Human > Assistant

- **Prompt safety**
  - Reduces accidental instruction override
  - Minimizes prompt injection risks

- **Consistency**
  - Same conversation structure across requests

- **Observability**
  - Each message can be logged, traced, replayed

---

## 6️⃣ ChatPromptTemplate vs PromptTemplate

### PromptTemplate
- Single block of text
- Best for:
  - Classification
  - Summarization
  - Extraction
  - RAG context formatting

### ChatPromptTemplate
- Structured message list
- Best for:
  - Chatbots
  - Assistants
  - Multi-turn conversations
  - Instruction-following systems

👉 ChatPromptTemplate is **higher-level**, but PromptTemplate still exists underneath.

---

## 7️⃣ Where ChatPromptTemplate Fits in Modern LangChain

Modern LangChain stack:

- PromptTemplate → deterministic text blocks  
- ChatPromptTemplate → structured chat messages  
- LLM / Chat Model → executes messages  
- LCEL (pipe operator) → connects everything cleanly  

ChatPromptTemplate is the **preferred interface for chat-based models**.

---

## 8️⃣ How Invocation Works (Conceptually)

Execution flow:

1. Inputs are provided once  
2. ChatPromptTemplate formats messages  
3. Messages are sent to the chat model  
4. The model generates a response  

Using LCEL allows this to happen in **one readable step** without glue code.

---

## 9️⃣ Why ChatPromptTemplate Is Important for Safety

Industry systems use ChatPromptTemplate to:

- Lock critical instructions into system messages  
- Prevent user input from overriding system rules  
- Separate policy from user intent  
- Reduce jailbreak and injection risks  

This is **mandatory** in regulated or customer-facing systems.

---

## 🔟 Typical Industry Use Cases

ChatPromptTemplate is commonly used in:

- Customer support bots  
- Internal copilots  
- AI tutors and assistants  
- Agent-based systems  
- RAG chat interfaces  

Anywhere **conversation matters**, ChatPromptTemplate is preferred.

---

## 1️⃣1️⃣ When ChatPromptTemplate Is Usually NOT Used

ChatPromptTemplate may be skipped when:

- Doing one-shot classification  
- Performing offline batch processing  
- Formatting retrieved documents  
- Running simple, non-interactive tasks  

In these cases, PromptTemplate is simpler and sufficient.

---

## 1️⃣2️⃣ Common Mistakes to Avoid

- Treating chat prompts as plain strings  
- Putting business logic in user messages  
- Mixing system and human instructions  
- Ignoring role priority  
- Using ChatPromptTemplate for non-chat tasks  

---

## 🧠 Final Mental Model

> **PromptTemplate builds text.  
> ChatPromptTemplate builds conversations.  
> Chat models reason over messages, not strings.**

---

## TL;DR

- ChatPromptTemplate = **conversation contract**
- Designed for **chat-native LLMs**
- Enforces role separation and safety
- Preferred for assistants and chat UX
- Built on top of PromptTemplate concepts

---
